# TradeSense · Component 03 · Layer 1 for August to October 2024, built from individual trades (v1)

**Why this notebook.** Notebook 23 showed that the stored Layer 1 counts *individual trades* (`trades_count`, `large_trade_count` per individual trade of at least 0.1 BTC / 2.0 ETH): it reproduces the Tardis raw-trade files to 99.9 to 100 percent. The Binance `aggTrades` files used in notebook 23 for the days between the first-of-month days merge fills, so their trade counts and large-trade counts are *not* the same quantity (compare 2024-10-01 Tardis 5.6 million BTC trades with 1.8 million aggTrades on 10-02). This notebook therefore rebuilds every day of 2024-08-01 .. 2024-10-31 from the **individual-trade** files of Binance (`spot/daily/trades`), checks the path against the Tardis files on the first-of-month days, and writes the contract features.

**Safe and resumable.** Existing Layer 1 and notebook 23 outputs are only read. Output: `layer1_rebuild_v1/bars_trades/` (one file per day) and `layer1_aug_oct_v1/` (feature matrices). The notebook stops itself after `MAX_RUN_MINUTES` and continues on the next *Run all*. CPU runtime; roughly 1 to 3 hours in total (a day is 0.5 to 1 minute).

In [1]:
import os, io, re, json, time, glob, zipfile, hashlib, warnings, urllib.request
from pathlib import Path
from datetime import date, timedelta
import numpy as np, pandas as pd
warnings.filterwarnings('ignore')
try:
    from google.colab import drive
    drive.mount('/content/drive')
except ImportError:
    print('Not in Colab -> using TRADESENSE_BASE.')
BASE = Path(os.environ.get('TRADESENSE_BASE', '/content/drive/MyDrive/research datasets')); assert BASE.is_dir(), BASE
R = BASE / 'layer1_rebuild_v1'; TB = R / 'bars_trades'; TB.mkdir(parents=True, exist_ok=True); (R/'zips').mkdir(exist_ok=True)
OUT = BASE / 'layer1_aug_oct_v1'; OUT.mkdir(exist_ok=True)
SYMBOLS = ['BTCUSDT', 'ETHUSDT']
FLOOR = {'BTCUSDT': 2.5399, 'ETHUSDT': 27.7658}
LARGE = {'BTCUSDT': 0.1, 'ETHUSDT': 2.0}
START, END = date(2024,8,1), date(2024,10,31)
MAX_RUN_MINUTES = float(os.environ.get('MAX_RUN_MINUTES', 150))
URL = os.environ.get('TRADES_URL', 'https://data.binance.vision/data/spot/daily/trades/{s}/{s}-trades-{d}.zip')
T_START = time.time(); LINES = []
def say(s=''): print(s); LINES.append(str(s))

Mounted at /content/drive


## 1 · Contract functions (same as notebook 23)

In [2]:
# 2. the frozen feature contract, applied to one continuous run of 1-minute bars
def bars_from_trades(ts_ms, price, qty, buyer_maker, first_id=None, last_id=None, large=0.1):
    """1-minute bars from (agg)trades. taker BUY volume = trades with is_buyer_maker == False."""
    ts_ms = np.asarray(ts_ms, dtype=np.int64)
    ts_ms = np.where(ts_ms > 10**14, ts_ms // 1000, ts_ms)           # microseconds -> ms (newer files)
    minute = ts_ms // 60000
    df = pd.DataFrame({'m': minute, 'p': price, 'q': qty, 'bm': buyer_maker})
    df['buy'] = np.where(~df.bm, df.q, 0.0); df['sell'] = np.where(df.bm, df.q, 0.0)
    df['big'] = (df.q >= large).astype(np.int64)
    if first_id is not None: df['ids'] = np.asarray(last_id) - np.asarray(first_id) + 1
    g = df.groupby('m', sort=True)
    b = pd.DataFrame({'open': g.p.first(), 'high': g.p.max(), 'low': g.p.min(), 'close': g.p.last(),
                      'volume': g.q.sum(), 'buy_volume': g.buy.sum(), 'sell_volume': g.sell.sum(),
                      'agg_count': g.q.size(), 'large_trade_count': g.big.sum()})
    b['trade_id_count'] = g.ids.sum() if first_id is not None else np.nan
    full = np.arange(b.index.min(), b.index.max()+1)
    b = b.reindex(full)
    b['close'] = b.close.ffill();
    for c in ['open','high','low']: b[c] = b[c].fillna(b.close)
    for c in ['volume','buy_volume','sell_volume','agg_count','large_trade_count','trade_id_count']: b[c] = b[c].fillna(0)
    b.index = pd.to_datetime(b.index*60000, unit='ms', utc=True); b.index.name = 'datetime'
    return b

def contract_features(b, symbol, trades_col='agg_count'):
    """Six Layer 1 features. Bars 0..14 of a run: volume_acceleration = 0.0 ; log_returns bar0 = 0."""
    o = pd.DataFrame(index=b.index)
    c = b.close.to_numpy(); v = b.volume.to_numpy()
    lr = np.zeros(len(b)); lr[1:] = np.log(c[1:]/c[:-1]); o['log_returns'] = lr
    o['trade_flow_imbalance'] = (b.buy_volume-b.sell_volume)/(b.buy_volume+b.sell_volume+1e-8)
    o['hl_spread_ratio'] = (b.high-b.low)/b.close
    o['rush_order_density'] = b[trades_col]/(b.volume+1e-8)
    va = np.zeros(len(b)); s = pd.Series(v)
    mu = s.shift(1).rolling(15).mean().to_numpy(); sd = s.shift(1).rolling(15).std(ddof=1).to_numpy()
    ok = np.arange(len(b)) >= 15
    va[ok] = (v[ok]-mu[ok])/np.maximum(sd[ok], FLOOR[symbol]); o['volume_acceleration'] = va
    o['large_trade_count'] = b.large_trade_count
    return o

def features_by_runs(b, symbol, trades_col='agg_count'):
    """Split at gaps in the day list (a missing day starts a new run -> warm-up masked again)."""
    gap = b.index.to_series().diff() != pd.Timedelta('1min'); rid = gap.cumsum().to_numpy()
    parts = [contract_features(b[rid==k], symbol, trades_col) for k in np.unique(rid)]
    return pd.concat(parts)

# unit tests on synthetic trades
rng = np.random.default_rng(0); n = 6000
t0 = 1714521600000; ts = np.sort(t0 + rng.integers(0, 40*60000, n))
px = 60000 + np.cumsum(rng.normal(0, 2, n)); q = rng.exponential(0.05, n); bm = rng.random(n) < 0.5
b = bars_from_trades(ts, px, q, bm, np.arange(n), np.arange(n)+rng.integers(0,3,n), large=0.1)
assert len(b) == 40 and abs(b.volume.sum()-q.sum()) < 1e-9 and abs(b.buy_volume.sum()+b.sell_volume.sum()-q.sum()) < 1e-9
f = contract_features(b, 'BTCUSDT'); assert (f.volume_acceleration.iloc[:15] == 0).all() and f.log_returns.iloc[0] == 0
i = 20; w = b.volume.iloc[i-15:i].to_numpy(); ref = (b.volume.iloc[i]-w.mean())/max(w.std(ddof=1), FLOOR['BTCUSDT'])
assert abs(f.volume_acceleration.iloc[i]-ref) < 1e-12
assert f.trade_flow_imbalance.between(-1,1).all() and (f.large_trade_count >= 0).all()
print('unit tests passed (bars, buy/sell split, vol_acc contract, warm-up mask)')

unit tests passed (bars, buy/sell split, vol_acc contract, warm-up mask)


## 2 · Individual-trade day files

In [3]:
# individual-trade day files -> bars (Tardis file in Drive is used for the first-of-month days: same quantity, no download)
tardis = []
for dp, dn, fn in os.walk(BASE):
    if str(Path(dp)).count(os.sep) - str(BASE).count(os.sep) > 4: dn[:] = []; continue
    for f in fn:
        if f.startswith('binance_trades_') and f.endswith('.csv.gz'): tardis.append(str(Path(dp)/f))
def bp(s, d): return TB/f'{s}_{d.isoformat()}_bars.parquet'
def one_day(s, d):
    p = bp(s, d)
    if p.exists(): return 'cached'
    tf = [x for x in tardis if s in x and d.isoformat() in Path(x).name]
    if tf:
        df = pd.read_csv(tf[0], usecols=['timestamp','side','price','amount','id'])
        ts, px, q, bm, ids = df.timestamp.to_numpy(), df.price.to_numpy(), df.amount.to_numpy(), (df.side=='sell').to_numpy(), df['id'].to_numpy(); src = 'Tardis'
    else:
        zp = R/'zips'/f'{s}-trades-{d.isoformat()}.zip'
        if not zp.exists(): urllib.request.urlretrieve(URL.format(s=s, d=d.isoformat()), zp)
        with zipfile.ZipFile(zp) as z, z.open(z.namelist()[0]) as fh:
            df = pd.read_csv(fh, header=None, usecols=[0,1,2,4,5])
        df.columns = ['id','price','qty','time','bm']
        ts, px, q, bm, ids = df.time.to_numpy(), df.price.to_numpy(), df.qty.to_numpy(), df.bm.astype(bool).to_numpy(), df['id'].to_numpy(); src = 'Binance trades'
        zp.unlink(missing_ok=True)
    b = bars_from_trades(ts, px, q, bm, ids, ids, large=LARGE[s])
    t0 = pd.Timestamp(d, tz='UTC'); b = b[(b.index >= t0) & (b.index < t0 + pd.Timedelta('1D'))]
    b.to_parquet(p); return f'ok ({src}) {len(b)} bars, {len(ids):,} trades'
days = [START + timedelta(days=i) for i in range((END-START).days+1)]
for d in days:
    for s in SYMBOLS:
        if bp(s, d).exists(): continue
        if (time.time()-T_START)/60 > MAX_RUN_MINUTES: break
        t = time.time()
        try: say(f'{s} {d}: {one_day(s, d)} ({time.time()-t:.0f}s)')
        except Exception as e: say(f'{s} {d}: FAILED {type(e).__name__} {e}')
    else: continue
    say(f'time budget reached after {(time.time()-T_START)/60:.0f} min -> run all again to continue'); break
have = sum(bp(s, d).exists() for d in days for s in SYMBOLS)
say(f'trade-based day files: {have} of {len(days)*len(SYMBOLS)}')

BTCUSDT 2024-08-01: ok (Tardis) 1440 bars, 2,397,256 trades (3s)
ETHUSDT 2024-08-01: ok (Tardis) 1440 bars, 1,835,455 trades (3s)
BTCUSDT 2024-08-02: ok (Binance trades) 1440 bars, 2,457,735 trades (3s)
ETHUSDT 2024-08-02: ok (Binance trades) 1440 bars, 2,104,725 trades (3s)
BTCUSDT 2024-08-03: ok (Binance trades) 1440 bars, 1,669,555 trades (2s)
ETHUSDT 2024-08-03: ok (Binance trades) 1440 bars, 1,798,017 trades (2s)
BTCUSDT 2024-08-04: ok (Binance trades) 1440 bars, 1,886,195 trades (3s)
ETHUSDT 2024-08-04: ok (Binance trades) 1440 bars, 2,184,760 trades (2s)
BTCUSDT 2024-08-05: ok (Binance trades) 1440 bars, 7,568,497 trades (7s)
ETHUSDT 2024-08-05: ok (Binance trades) 1440 bars, 7,064,159 trades (9s)
BTCUSDT 2024-08-06: ok (Binance trades) 1440 bars, 3,433,824 trades (4s)
ETHUSDT 2024-08-06: ok (Binance trades) 1440 bars, 2,870,150 trades (3s)
BTCUSDT 2024-08-07: ok (Binance trades) 1440 bars, 2,397,504 trades (3s)
ETHUSDT 2024-08-07: ok (Binance trades) 1440 bars, 2,370,148 trades

## 3 · aggTrades versus individual trades

In [4]:
# cross-check 1: individual-trade path vs Tardis on a first-of-month day is only possible when both exist -> use aggTrades vs trades on the same day to show the difference
rows = []
for s in SYMBOLS:
    for d in [date(2024,8,5), date(2024,9,10), date(2024,10,15)]:
        a = R/'bars'/f'{s}_{d.isoformat()}_bars.parquet'; t = bp(s, d)
        if a.exists() and t.exists():
            A, T = pd.read_parquet(a), pd.read_parquet(t)
            rows.append((s, str(d), float(np.max(np.abs(A.volume-T.volume))), int(A.agg_count.sum()), int(T.trade_id_count.sum()), int(A.large_trade_count.sum()), int(T.large_trade_count.sum())))
if rows:
    df = pd.DataFrame(rows, columns=['symbol','day','max|volume diff|','aggTrades count','individual trades','large (aggTrades)','large (individual)'])
    say(df.to_string(index=False))
    say('Volume must agree; the counts differ, which is why the individual-trade files are the source of the Layer 1 contract for these days.')
else: say('(no day available in both sources yet)')

 symbol        day  max|volume diff|  aggTrades count  individual trades  large (aggTrades)  large (individual)
BTCUSDT 2024-08-05      2.273737e-13          5280704            7568497             389140              392360
BTCUSDT 2024-09-10      1.421085e-14          1167016            2843148              43994               48792
BTCUSDT 2024-10-15      2.842171e-14          2046567            6245619              88903              101722
ETHUSDT 2024-08-05      4.547474e-13          4964991            7064159             357917              319214
ETHUSDT 2024-09-10      4.547474e-13           642920            1210860              26386               29903
ETHUSDT 2024-10-15      2.273737e-13           905395            3306565              50720               52529
Volume must agree; the counts differ, which is why the individual-trade files are the source of the Layer 1 contract for these days.


## 4 · Contract features and drift

In [5]:
# contract features: trades_col = agg_count here means rows of the trade files = individual trades (Tardis rows or Binance trade ids)
COLS = ['log_returns','trade_flow_imbalance','hl_spread_ratio','rush_order_density','volume_acceleration','large_trade_count']
report = {}
for s in SYMBOLS:
    ps = [bp(s, d) for d in days if bp(s, d).exists()]
    if len(ps) < len(days): say(f'{s}: only {len(ps)} of {len(days)} days -> features not written yet'); continue
    b = pd.concat([pd.read_parquet(p) for p in ps]).sort_index(); assert b.index.is_unique and b.index.is_monotonic_increasing
    assert (b.index.to_series().diff().dropna() == pd.Timedelta('1min')).all(), 'gap in bars'
    f = contract_features(b, s, 'trade_id_count')
    out = b[['open','high','low','close','volume']].copy(); out['trades_count'] = b.trade_id_count
    for c in COLS: out[c] = f[c]
    out['warmup_masked'] = np.arange(len(out)) < 15
    out.to_parquet(OUT/f'{s.lower()}_l1_feature_matrix_aug_oct_2024.parquet')
    ex = pd.read_parquet(sorted(glob.glob(str(BASE/'**'/f'{s.lower()}_l1_feature_matrix_90d.parquet'), recursive=True))[0])
    if 'datetime' in ex.columns: ex = ex.set_index('datetime')
    rep = {'rows': len(out), 'first': str(out.index[0]), 'last': str(out.index[-1]), 'nan': int(out[COLS].isna().sum().sum()),
           'vol_acc_min': float(out.volume_acceleration.min()), 'vol_acc_max': float(out.volume_acceleration.max())}
    # drift of every feature against the old Layer 1 training split (median and IQR), so we know how different the test months look
    for c in COLS:
        a, e = out[c].to_numpy()[15:], ex[c].to_numpy()[15:]
        rep[c] = {'median_new': float(np.median(a)), 'median_L1': float(np.median(e)), 'iqr_new': float(np.subtract(*np.percentile(a,[75,25]))), 'iqr_L1': float(np.subtract(*np.percentile(e,[75,25])))}
    report[s] = rep; say(f'{s}: {rep["rows"]} bars {rep["first"]} .. {rep["last"]}  NaN {rep["nan"]}  vol_acc range {rep["vol_acc_min"]:.2f} .. {rep["vol_acc_max"]:.2f}')
    for c in COLS: say(f'    {c:22s} median new {rep[c]["median_new"]:11.5g} (Layer 1 {rep[c]["median_L1"]:11.5g})   IQR new {rep[c]["iqr_new"]:10.5g} (Layer 1 {rep[c]["iqr_L1"]:10.5g})')

BTCUSDT: 132480 bars 2024-08-01 00:00:00+00:00 .. 2024-10-31 23:59:00+00:00  NaN 0  vol_acc range -3.46 .. 237.45
    log_returns            median new           0 (Layer 1           0)   IQR new 0.00058945 (Layer 1 0.00053949)
    trade_flow_imbalance   median new   -0.045283 (Layer 1   -0.030863)   IQR new    0.73313 (Layer 1    0.73664)
    hl_spread_ratio        median new  0.00050225 (Layer 1    0.000453)   IQR new 0.00064338 (Layer 1 0.00058539)
    rush_order_density     median new      136.72 (Layer 1      62.151)   IQR new     138.79 (Layer 1     49.606)
    volume_acceleration    median new    -0.31967 (Layer 1     -0.3148)   IQR new     1.0041 (Layer 1     1.1041)
    large_trade_count      median new          20 (Layer 1          25)   IQR new         36 (Layer 1         41)
ETHUSDT: 132480 bars 2024-08-01 00:00:00+00:00 .. 2024-10-31 23:59:00+00:00  NaN 0  vol_acc range -4.08 .. 231.82
    log_returns            median new           0 (Layer 1           0)   IQR new 0.0007

## 5 · Share block

In [6]:
(OUT/'aug_oct_report.json').write_text(json.dumps(report, indent=1)); block = '\n'.join(LINES); (OUT/'aug_oct_share_block.txt').write_text(block)
print('\n===== SHARE BLOCK (copy everything below, or attach layer1_aug_oct_v1/aug_oct_share_block.txt) =====\n' + block[-5000:])


===== SHARE BLOCK (copy everything below, or attach layer1_aug_oct_v1/aug_oct_share_block.txt) =====
nce trades) 1440 bars, 6,245,619 trades (6s)
ETHUSDT 2024-10-15: ok (Binance trades) 1440 bars, 3,306,565 trades (4s)
BTCUSDT 2024-10-16: ok (Binance trades) 1440 bars, 5,109,685 trades (5s)
ETHUSDT 2024-10-16: ok (Binance trades) 1440 bars, 2,813,136 trades (3s)
BTCUSDT 2024-10-17: ok (Binance trades) 1440 bars, 3,981,960 trades (4s)
ETHUSDT 2024-10-17: ok (Binance trades) 1440 bars, 2,543,113 trades (3s)
BTCUSDT 2024-10-18: ok (Binance trades) 1440 bars, 4,010,969 trades (4s)
ETHUSDT 2024-10-18: ok (Binance trades) 1440 bars, 2,835,054 trades (3s)
BTCUSDT 2024-10-19: ok (Binance trades) 1440 bars, 1,152,428 trades (2s)
ETHUSDT 2024-10-19: ok (Binance trades) 1440 bars, 1,145,594 trades (2s)
BTCUSDT 2024-10-20: ok (Binance trades) 1440 bars, 1,563,795 trades (2s)
ETHUSDT 2024-10-20: ok (Binance trades) 1440 bars, 2,274,845 trades (3s)
BTCUSDT 2024-10-21: ok (Binance trades) 1440 bars,